# 02a — Prétraitement des Modèles Numériques de Terrain (MNT/DEM)

**Département** : Hérault (34), France  
**Emprise** : `[2.85, 43.21, 3.98, 43.95]` (lon_min, lat_min, lon_max, lat_max)  
**Objectif** : Télécharger, conditionner et analyser les MNTs pour la modélisation des crues.

---

## Plan
1. **Copernicus DEM GLO-30** — données mondiales à 30m
2. **IGN RGE ALTI / LiDAR HD** — données France haute résolution (1m / 50cm)
3. **Prétraitement pysheds** — conditionnement hydrologique, directions d'écoulement
4. **HAND** — Height Above Nearest Drainage, proxy d'inondabilité
5. **Whitebox Tools** — outils complémentaires (TWI, D-inf, breach depressions)
6. **Synthèse** — comparaison des sources et méthodes

In [1]:
# ─── Imports et constantes globales ───────────────────────────────────────────
import os
import warnings
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource
from pathlib import Path

warnings.filterwarnings('ignore')

# Constantes du projet
DEPT_CODE = "34"
DEPT_NAME = "Hérault"
BBOX      = [2.85, 43.21, 3.98, 43.95]   # [lon_min, lat_min, lon_max, lat_max]
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = BBOX

DATA_DIR = Path("../../data/terrain")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Département : {DEPT_NAME} ({DEPT_CODE})")
print(f"BBOX        : {BBOX}")
print(f"Dossier     : {DATA_DIR.resolve()}")

Département : Hérault (34)
BBOX        : [2.85, 43.21, 3.98, 43.95]
Dossier     : /Users/alexandre/Documents/Geospatial/floods-france/data/terrain


---
## Section 1 — Copernicus DEM GLO-30 (30 m)

Le **Copernicus DEM GLO-30** est un MNT mondial à résolution ~30m dérivé des données TanDEM-X de la mission Copernicus. Il est disponible gratuitement via :
- **OpenTopography API** (accès direct, clé API requise)
- **py3dep** (wrapper Python pour l'API 3DEP / OpenTopography)
- **Planetary Computer STAC** (Microsoft, accès anonyme)

> **Référence** : Copernicus DEM — European Space Agency, © DLR e.V. 2010-2011 and © Airbus Defence and Space GmbH 2014-2018.

### 1.1 — OpenTopography API

In [2]:
# ─── Construction de l'URL OpenTopography ─────────────────────────────────────
OT_BASE  = "https://portal.opentopography.org/API/globaldem"
OT_PARAMS = {
    "demtype"     : "COP30",
    "south"       : LAT_MIN,
    "north"       : LAT_MAX,
    "west"        : LON_MIN,
    "east"        : LON_MAX,
    "outputFormat": "GTiff",
    "API_Key"     : "YOUR_KEY",   # ← remplacer par votre clé OpenTopography
}

# URL complète affichée pour documentation
OT_URL = (
    f"{OT_BASE}"
    f"?demtype={OT_PARAMS['demtype']}"
    f"&south={OT_PARAMS['south']}"
    f"&north={OT_PARAMS['north']}"
    f"&west={OT_PARAMS['west']}"
    f"&east={OT_PARAMS['east']}"
    f"&outputFormat={OT_PARAMS['outputFormat']}"
    f"&API_Key={OT_PARAMS['API_Key']}"
)
print("URL OpenTopography COP30 :")
print(OT_URL)

URL OpenTopography COP30 :
https://portal.opentopography.org/API/globaldem?demtype=COP30&south=43.21&north=43.95&west=2.85&east=3.98&outputFormat=GTiff&API_Key=YOUR_KEY


In [3]:
# ─── Téléchargement via OpenTopography (nécessite une clé API valide) ─────────
DEM_PATH = DATA_DIR / "dem_herault_cop30.tif"

def download_dem_opentopography(url: str, dest: Path, api_key: str = "YOUR_KEY") -> bool:
    """
    Télécharge le COP30 via l'API OpenTopography.
    Retourne True si le téléchargement est réussi, False sinon.
    """
    if dest.exists():
        print(f"Fichier existant : {dest}")
        return True
    if api_key == "YOUR_KEY":
        print("[SKIP] Clé API OpenTopography non renseignée.")
        print("       Obtenez une clé gratuite sur : https://portal.opentopography.org/")
        return False
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        print(f"Téléchargé : {dest} ({dest.stat().st_size / 1e6:.1f} Mo)")
        return True
    except requests.RequestException as e:
        print(f"[ERREUR] Téléchargement échoué : {e}")
        return False

ot_ok = download_dem_opentopography(OT_URL, DEM_PATH)

[SKIP] Clé API OpenTopography non renseignée.
       Obtenez une clé gratuite sur : https://portal.opentopography.org/


### 1.2 — Alternatives : py3dep et Planetary Computer STAC

In [ ]:
# ─── Alternative 1 : py3dep ───────────────────────────────────────────────────
# py3dep encapsule l'accès aux DEMs mondiaux via l'API 3DEP/OpenTopography.
# Installation : pip install py3dep
#
# import py3dep
# dem = py3dep.get_dem(
#     geometry=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
#     resolution=30,
#     crs="EPSG:4326",
#     dem_name="COP30",
# )
# # dem est un xarray.DataArray — sauvegarde directe
# dem.rio.to_raster(DATA_DIR / "dem_herault_py3dep.tif")
print("[py3dep] Exemple de code disponible en commentaire ci-dessus.")
print("         Nécessite : pip install py3dep")

In [ ]:
# ─── Alternative 2 : Planetary Computer STAC (Microsoft) ─────────────────────
# Accès gratuit et anonyme au Copernicus DEM via le catalogue STAC de Microsoft.
# Installation : pip install pystac-client planetary-computer rioxarray
#
# import pystac_client
# import planetary_computer
# import rioxarray as rxr
#
# catalog = pystac_client.Client.open(
#     "https://planetarycomputer.microsoft.com/api/stac/v1",
#     modifier=planetary_computer.sign_inplace,
# )
# search = catalog.search(
#     collections=["cop-dem-glo-30"],
#     bbox=BBOX,
# )
# items = list(search.get_all_items())
# print(f"{len(items)} tuile(s) COP30 trouvée(s)")
#
# # Lecture directe via rioxarray + stackstac
# import stackstac
# ds = stackstac.stack(
#     items,
#     assets=["data"],
#     bounds_latlon=BBOX,
# )
# dem_stac = ds.squeeze().compute()
print("[Planetary Computer] Exemple de code disponible en commentaire ci-dessus.")
print("                     Nécessite : pip install pystac-client planetary-computer stackstac")

### 1.3 — Chargement avec rioxarray et statistiques de base

In [5]:
!pip install rioxarray

  Using cached rioxarray-0.19.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached rasterio-1.4.4-cp310-cp310-macosx_14_0_arm64.whl.metadata (9.3 kB)
  Using cached xarray-2025.6.1-py3-none-any.whl.metadata (12 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached rioxarray-0.19.0-py3-none-any.whl (62 kB)
Using cached rasterio-1.4.4-cp310-cp310-macosx_14_0_arm64.whl (21.1 MB)
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
Using cached xarray-2025.6.1-py3-none-any.whl (1.3 MB)
Using cached affine-2.4.0-py3-none-any.whl (15 kB)
Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [rioxarray]/6 [xarray]o]


In [6]:
# ─── Chargement du DEM avec rioxarray ─────────────────────────────────────────
# Installation : pip install rioxarray
try:
    import rioxarray as rxr

    dem_path_str = str(DEM_PATH)
    # Chargement standard
    dem = rxr.open_rasterio('dem_herault.tif')  # remplacer par le chemin réel
    # dem = rxr.open_rasterio(dem_path_str)      # ← utiliser cette ligne en production

    print("Dimensions :", dem.dims)
    print("CRS        :", dem.rio.crs)
    print("Résolution :", dem.rio.resolution())
    print("Bounds     :", dem.rio.bounds())
    print("Shape      :", dem.shape)

except FileNotFoundError:
    print("[INFO] Fichier DEM introuvable — génération d'un DEM synthétique pour la démonstration.")
    # DEM synthétique représentant le relief héraultais (Plaine côtière + Cévennes)
    ny, nx = 300, 400
    x = np.linspace(LON_MIN, LON_MAX, nx)
    y = np.linspace(LAT_MIN, LAT_MAX, ny)
    X, Y = np.meshgrid(x, y)
    # Gradient nord-sud simulant la montée cévenole + plaine littorale
    dem_arr = (
        50
        + 600 * np.clip((Y - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1) ** 2
        + 200 * np.sin(3 * (X - LON_MIN) / (LON_MAX - LON_MIN) * np.pi)
             * np.clip((Y - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1)
        + 30  * np.random.RandomState(42).randn(ny, nx)
    ).clip(0, 1200)
    dem_data = dem_arr
    print(f"DEM synthétique généré : shape={dem_data.shape}")
except ImportError:
    print("[INFO] rioxarray non installé — génération d'un DEM synthétique.")
    ny, nx = 300, 400
    x = np.linspace(LON_MIN, LON_MAX, nx)
    y = np.linspace(LAT_MIN, LAT_MAX, ny)
    X, Y = np.meshgrid(x, y)
    dem_arr = (
        50
        + 600 * np.clip((Y - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1) ** 2
        + 200 * np.sin(3 * (X - LON_MIN) / (LON_MAX - LON_MIN) * np.pi)
             * np.clip((Y - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1)
        + 30  * np.random.RandomState(42).randn(ny, nx)
    ).clip(0, 1200)
    dem_data = dem_arr
    print(f"DEM synthétique généré : shape={dem_data.shape}")

RasterioIOError: dem_herault.tif: No such file or directory

In [ ]:
# ─── Statistiques de base du DEM ──────────────────────────────────────────────
# Si rioxarray disponible, utiliser dem.values[0]; sinon dem_data
try:
    arr = dem.values[0].astype(float)
    arr[arr < -1000] = np.nan  # masquage nodata
except NameError:
    arr = dem_data.copy()

print(f"{'Statistique':<20} {'Valeur':>12}")
print("-" * 34)
print(f"{'Altitude min (m)':<20} {np.nanmin(arr):>12.1f}")
print(f"{'Altitude max (m)':<20} {np.nanmax(arr):>12.1f}")
print(f"{'Altitude moyenne (m)':<20} {np.nanmean(arr):>12.1f}")
print(f"{'Écart-type (m)':<20} {np.nanstd(arr):>12.1f}")
print(f"{'Médiane (m)':<20} {np.nanmedian(arr):>12.1f}")
print(f"{'P10 (m)':<20} {np.nanpercentile(arr, 10):>12.1f}")
print(f"{'P90 (m)':<20} {np.nanpercentile(arr, 90):>12.1f}")

### 1.4 — Visualisation du DEM (carte + ombrage)

In [ ]:
# ─── Visualisation DEM avec effet hillshade ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Copernicus DEM GLO-30 — {DEPT_NAME} (dép. {DEPT_CODE})", fontsize=14, fontweight='bold')

extent = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]

# ── Panneau gauche : DEM coloré ────────────────────────────────────────────────
ax1 = axes[0]
cmap_dem = plt.cm.terrain
im1 = ax1.imshow(
    arr, extent=extent, origin='lower',
    cmap=cmap_dem, vmin=0, vmax=np.nanpercentile(arr, 99)
)
cb1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cb1.set_label("Altitude (m)", fontsize=10)
ax1.set_title("MNT — Copernicus COP30", fontsize=11)
ax1.set_xlabel("Longitude (°E)")
ax1.set_ylabel("Latitude (°N)")
ax1.grid(True, linestyle='--', alpha=0.4)

# Annotations géographiques principales
ax1.annotate("Montpellier", xy=(3.88, 43.61), fontsize=7, color='white',
             xytext=(3.6, 43.55), arrowprops=dict(arrowstyle='->', color='white', lw=0.8))
ax1.annotate("Hérault (fleuve)", xy=(3.2, 43.45), fontsize=7, color='cyan',
             xytext=(3.05, 43.35), arrowprops=dict(arrowstyle='->', color='cyan', lw=0.8))

# ── Panneau droit : Hillshade (ombrage) ────────────────────────────────────────
ax2 = axes[1]
ls = LightSource(azdeg=315, altdeg=45)  # azimut nord-ouest, angle 45°
arr_filled = np.where(np.isnan(arr), 0, arr)
hillshade = ls.hillshade(arr_filled, vert_exag=3, dx=30, dy=30)  # dx/dy en mètres

# Superposition couleur + hillshade
rgb = ls.shade(
    arr_filled, cmap=cmap_dem,
    blend_mode='overlay',
    vmin=0, vmax=np.nanpercentile(arr, 99),
    vert_exag=3, dx=30, dy=30
)
ax2.imshow(rgb, extent=extent, origin='lower')
ax2.set_title("Ombrage (hillshade, exag. ×3)", fontsize=11)
ax2.set_xlabel("Longitude (°E)")
ax2.set_ylabel("Latitude (°N)")
ax2.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
fig.savefig(DATA_DIR / "dem_cop30_visualisation.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée.")

---
## Section 2 — IGN RGE ALTI (1 m) et LiDAR HD (50 cm)

L'**IGN** (Institut national de l'information géographique et forestière) propose deux sources haute résolution :

| Source | Résolution | Couverture | Accès |
|---|---|---|---|
| **RGE ALTI®** | 1 m | France entière | WCS Géoportail (gratuit) |
| **LiDAR HD** | 50 cm | En cours (2020–2025) | Diffusion IGN (gratuit) |

### 2.1 — Web Coverage Service (WCS) IGN

In [ ]:
# ─── Endpoint WCS IGN ─────────────────────────────────────────────────────────
IGN_WCS_BASE = "https://wxs.ign.fr/altimetrie/geoportail/r/wcs"

# ── GetCapabilities : liste les couvertures disponibles ───────────────────────
get_cap_url = f"{IGN_WCS_BASE}?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCapabilities"
print("URL GetCapabilities :")
print(get_cap_url)
print()

try:
    resp_cap = requests.get(get_cap_url, timeout=30)
    if resp_cap.status_code == 200:
        print(f"Réponse reçue : {resp_cap.status_code} — {len(resp_cap.content)} octets")
        # Aperçu des 500 premiers caractères
        print(resp_cap.text[:500])
    else:
        print(f"Statut HTTP : {resp_cap.status_code}")
except requests.RequestException as e:
    print(f"[INFO] Service WCS IGN non joignable : {e}")
    print("       Vérifiez votre connexion ou réessayez ultérieurement.")

In [ ]:
# ─── GetCoverage : téléchargement d'une tuile autour de Montpellier ───────────
# BBOX de démonstration : zone restreinte autour de Montpellier
MONTPELLIER_BBOX = [3.8, 43.5, 3.95, 43.65]  # [lon_min, lat_min, lon_max, lat_max]

# L'ID de couverture RGE ALTI 1m est "RGEALTI_PYR-ZIP_FXX_Lambert93_WGS84G_2_0"
# ou simplement 'ELEVATION.ELEVATIONGRIDCOVERAGE' selon le serveur
COVERAGE_ID = "ELEVATION.ELEVATIONGRIDCOVERAGE.HIGHRES"

get_cov_url = (
    f"{IGN_WCS_BASE}"
    f"?SERVICE=WCS"
    f"&VERSION=2.0.1"
    f"&REQUEST=GetCoverage"
    f"&CoverageID={COVERAGE_ID}"
    f"&SUBSET=Long({MONTPELLIER_BBOX[0]},{MONTPELLIER_BBOX[2]})"
    f"&SUBSET=Lat({MONTPELLIER_BBOX[1]},{MONTPELLIER_BBOX[3]})"
    f"&format=image/tiff"
)

print("URL GetCoverage (RGE ALTI 1m — Montpellier) :")
print(get_cov_url)
print()

ign_dest = DATA_DIR / "dem_montpellier_rge1m.tif"
if not ign_dest.exists():
    try:
        resp_cov = requests.get(get_cov_url, timeout=120)
        if resp_cov.status_code == 200 and len(resp_cov.content) > 10_000:
            ign_dest.write_bytes(resp_cov.content)
            print(f"Tuile RGE ALTI sauvegardée : {ign_dest} ({ign_dest.stat().st_size / 1e6:.2f} Mo)")
        else:
            print(f"[INFO] Réponse WCS non valide (statut {resp_cov.status_code}, {len(resp_cov.content)} octets).")
            print("       Le service peut nécessiter une authentification ou un CoverageID différent.")
    except requests.RequestException as e:
        print(f"[INFO] Téléchargement WCS échoué : {e}")
else:
    print(f"Tuile déjà présente : {ign_dest}")

### 2.2 — LiDAR HD IGN (50 cm)

In [ ]:
# ─── LiDAR HD IGN — accès et téléchargement ───────────────────────────────────
# Le programme LiDAR HD vise la couverture totale de la France métropolitaine
# à 50 cm de résolution (terminé ~2025).
# Les données sont distribuées en dalles de 1 km × 1 km au format LAZ/COPC.

LIDAR_HD_BASE = "https://diffusion.ign.fr/LIDARHD/"
print("Portail LiDAR HD IGN :")
print(LIDAR_HD_BASE)
print()
print("Format des dalles : LAZ (LAS compressé) / COPC (Cloud Optimized Point Cloud)")
print("Emprise d'une dalle : 1 km × 1 km")
print("Résolution grille MNT dérivé : 0.5 m")
print()

# Exemple de téléchargement programmatique d'une dalle LiDAR HD
def download_lidar_hd_tile(tile_url: str, dest: Path) -> bool:
    """
    Télécharge une dalle LiDAR HD depuis le serveur IGN.
    
    Paramètres
    ----------
    tile_url : str
        URL complète de la dalle (format .laz ou .copc.laz)
    dest : Path
        Chemin de destination local
    """
    if dest.exists():
        print(f"Dalle existante : {dest}")
        return True
    try:
        print(f"Téléchargement : {tile_url}")
        resp = requests.get(tile_url, timeout=300, stream=True)
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        downloaded = 0
        with open(dest, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
        print(f"Sauvegardé : {dest} ({downloaded / 1e6:.1f} Mo)")
        return True
    except requests.RequestException as e:
        print(f"[ERREUR] {e}")
        return False

# Exemple d'URL pour une dalle LiDAR HD en Hérault (format générique)
# Les URLs exactes sont accessibles via le géoportail IGN ou l'API de diffusion
example_tile_url = (
    "https://diffusion.ign.fr/LIDARHD/projet/"
    "Herault-2022/LHD_FXX_0838_6280_PTS_C_LAMB93_IGN69.copc.laz"
)
print(f"Exemple URL dalle : {example_tile_url}")
print("[SKIP] Téléchargement non effectué (démonstration uniquement).")

### 2.3 — Comparaison Copernicus 30m vs RGE ALTI 1m

In [ ]:
# ─── Tableau comparatif des sources DEM ───────────────────────────────────────
comparison_data = {
    "Caractéristique"    : ["Résolution spatiale", "Précision altimétrique", "Source",
                            "Couverture", "Format", "Accès", "Mise à jour",
                            "Usage principal"],
    "Copernicus GLO-30"  : ["~30 m", "±4 m (RMSE)", "TanDEM-X (radar SAR)",
                            "Mondial", "GeoTIFF", "Gratuit (API OpenTopography)",
                            "Statique (2010–2015)", "Analyse régionale, bassin versant"],
    "IGN RGE ALTI® 1m"   : ["1 m", "±0.2 m (zones ouvertes)", "LiDAR aérien + nivellement",
                            "France métropolitaine", "GeoTIFF / ASCII", "Gratuit (WCS Géoportail)",
                            "Continu (actualisation locale)", "Cartographie fine, hydraulique"],
    "IGN LiDAR HD 50cm" : ["0.5 m", "±0.10–0.15 m", "LiDAR aérien full-waveform",
                            "France (en cours)", "LAZ / COPC", "Gratuit (diffusion.ign.fr)",
                            "Campagnes 2020–2025", "Inondations, MNS, végétation"],
}

try:
    import pandas as pd
    df_comp = pd.DataFrame(comparison_data).set_index("Caractéristique")
    print(df_comp.to_string())
except ImportError:
    # Affichage manuel si pandas non disponible
    cols = list(comparison_data.keys())
    header = f"{cols[0]:<28} | {cols[1]:<32} | {cols[2]:<32} | {cols[3]:<32}"
    print(header)
    print("-" * len(header))
    n = len(comparison_data[cols[0]])
    for i in range(n):
        row = " | ".join(f"{comparison_data[c][i]:<31}" for c in cols)
        print(row)

---
## Section 3 — Prétraitement DEM avec pysheds

Le conditionnement hydrologique du MNT est une étape indispensable avant toute analyse d'écoulement. Les MNTs bruts contiennent des artefacts (puits, dépressions fermées, zones plates) qui bloquent le cheminement de l'eau.

**Chaîne de traitement pysheds :**
```
DEM brut → Remplissage puits → Remplissage dépressions → Résolution zones plates
        → Directions d'écoulement (D8) → Accumulation → Extraction réseau hydrographique
        → Délimitation bassin versant
```

> Installation : `pip install pysheds`

In [ ]:
# ─── Chargement du DEM avec pysheds ───────────────────────────────────────────
try:
    from pysheds.grid import Grid
    PYSHEDS_AVAILABLE = True
    print("pysheds importé avec succès.")
except ImportError:
    PYSHEDS_AVAILABLE = False
    print("[INFO] pysheds non installé. Exécutez : pip install pysheds")
    print("       Les cellules suivantes montrent le code mais utilisent des données synthétiques.")

In [ ]:
# ─── Génération/chargement du DEM de travail ──────────────────────────────────
# Pour la démonstration, on crée un DEM synthétique si le fichier réel est absent.
rng = np.random.RandomState(42)
ny_s, nx_s = 200, 260
xs = np.linspace(LON_MIN, LON_MAX, nx_s)
ys = np.linspace(LAT_MIN, LAT_MAX, ny_s)
Xs, Ys = np.meshgrid(xs, ys)

# Relief synthétique : plaine littorale au sud, relief cévenol au nord
dem_synth = (
    20
    + 800 * np.clip((Ys - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1) ** 1.8
    + 150 * np.sin(4 * np.pi * (Xs - LON_MIN) / (LON_MAX - LON_MIN))
          * np.clip((Ys - LAT_MIN) / (LAT_MAX - LAT_MIN), 0, 1)
    - 80  * np.exp(-((Xs - 3.2)**2 + (Ys - 43.55)**2) / 0.05)   # vallée Hérault
    - 40  * np.exp(-((Xs - 3.6)**2 + (Ys - 43.45)**2) / 0.02)   # plaine Montpellier
    + 15  * rng.randn(ny_s, nx_s)  # bruit
).clip(0, 1000)

# Introduction de quelques artefacts pour démonstration du conditionnement
dem_synth[80:85, 100:105]  = dem_synth[80:85, 100:105] - 50   # dépression artificielle
dem_synth[120:122, 60:62]  = dem_synth[120:122, 60:62]  - 30   # puits

print(f"DEM synthétique : shape={dem_synth.shape}, min={dem_synth.min():.1f}m, max={dem_synth.max():.1f}m")

# Sauvegarde temporaire en GeoTIFF si rasterio disponible
dem_temp_path = DATA_DIR / "dem_synth_herault.tif"
try:
    import rasterio
    from rasterio.transform import from_bounds
    transform = from_bounds(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, nx_s, ny_s)
    with rasterio.open(
        dem_temp_path, 'w',
        driver='GTiff', height=ny_s, width=nx_s,
        count=1, dtype='float32',
        crs='EPSG:4326', transform=transform
    ) as dst:
        dst.write(dem_synth.astype('float32'), 1)
    print(f"DEM sauvegardé : {dem_temp_path}")
except ImportError:
    print("[INFO] rasterio non installé — le fichier GeoTIFF ne sera pas sauvegardé.")

In [ ]:
# ─── Conditionnement hydrologique avec pysheds ────────────────────────────────
if PYSHEDS_AVAILABLE and dem_temp_path.exists():
    # Chargement via pysheds Grid
    grid = Grid.from_raster(str(dem_temp_path))
    dem_ps = grid.read_raster(str(dem_temp_path))
    print(f"DEM chargé dans pysheds : shape={dem_ps.shape}")

    # ── Étape 1 : Remplissage des puits (pit filling) ─────────────────────────
    dem_c = grid.fill_pits(dem_ps)
    n_pits = int(np.sum(dem_c.base > dem_ps.base) if hasattr(dem_c, 'base') else 0)
    print(f"Puits remplis — cellules modifiées : ~{n_pits}")

    # ── Étape 2 : Remplissage des dépressions (depression filling) ────────────
    dem_cd = grid.fill_depressions(dem_c)
    print("Dépressions remplies.")

    # ── Étape 3 : Résolution des zones plates (flats) ─────────────────────────
    dem_cdf = grid.resolve_flats(dem_cd)
    print("Zones plates résolues.")

    # ── Étape 4 : Directions d'écoulement D8 ─────────────────────────────────
    # Codage D8 : N=64, NE=128, E=1, SE=2, S=4, SO=8, O=16, NO=32
    dirmap = (64, 128, 1, 2, 4, 8, 16, 32)  # pysheds standard
    fdir = grid.flowdir(dem_cdf, dirmap=dirmap)
    print(f"Directions d'écoulement D8 calculées : shape={fdir.shape}")

    # ── Étape 5 : Accumulation de flux ───────────────────────────────────────
    acc = grid.accumulation(fdir, dirmap=dirmap)
    print(f"Accumulation calculée : max={acc.max():.0f} cellules")

    # ── Étape 6 : Extraction du réseau hydrographique ─────────────────────────
    STREAM_THRESHOLD = 1000  # nombre minimum de cellules amont
    streams = acc > STREAM_THRESHOLD
    n_stream_cells = int(streams.sum())
    print(f"Réseau hydrographique (seuil={STREAM_THRESHOLD}) : {n_stream_cells} cellules")

else:
    print("[INFO] Calcul pysheds simulé (bibliothèque non disponible ou fichier absent).")
    # Simulation des résultats pour la visualisation
    dem_cdf  = dem_synth.copy()
    # Accumulation synthétique : plus haute vers le bas du versant
    acc_base = np.zeros_like(dem_synth)
    for i in range(ny_s):
        acc_base[i, :] = (i / ny_s) * nx_s * 0.3
    acc = acc_base + rng.rand(ny_s, nx_s) * 50
    # Amplification dans la vallée synthétique
    valley_mask = dem_synth < np.percentile(dem_synth, 25)
    acc[valley_mask] *= 5
    STREAM_THRESHOLD = 500
    streams = acc > STREAM_THRESHOLD
    fdir = None
    print(f"Accumulation synthétique — max={acc.max():.0f}")

In [ ]:
# ─── Extraction du bassin versant (pour un exutoire donné) ────────────────────
if PYSHEDS_AVAILABLE and fdir is not None:
    # Point de fermeture (pour point) : embouchure de l'Hérault simulée
    pour_point_lon = 3.25
    pour_point_lat = 43.30
    try:
        x_snap, y_snap = grid.snap_to_mask(acc > 500, (pour_point_lon, pour_point_lat))
        catch = grid.catchment(x=x_snap, y=y_snap, fdir=fdir, dirmap=dirmap, xytype='coordinate')
        grid.clip_to(catch)
        catchment_arr = grid.view(catch)
        print(f"Bassin versant extrait : {catchment_arr.sum()} cellules")
        area_km2 = catchment_arr.sum() * (30 * 30) / 1e6
        print(f"Surface approximative : {area_km2:.1f} km²")
    except Exception as e:
        print(f"[INFO] Extraction bassin versant : {e}")
        catchment_arr = None
else:
    # Bassin versant synthétique
    catchment_arr = (dem_synth < np.percentile(dem_synth, 40)).astype(float)
    print("Bassin versant synthétique généré pour visualisation.")

In [ ]:
# ─── Visualisation : DEM conditionné, accumulation, réseau hydrographique ─────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Prétraitement hydrologique du MNT — pysheds", fontsize=14, fontweight='bold')

extent_s = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]
arr_dem  = dem_cdf if not hasattr(dem_cdf, 'base') else np.array(dem_cdf)
if hasattr(arr_dem, 'data'):  # xarray/pysheds Raster
    arr_dem = np.array(arr_dem)

# ── (a) DEM conditionné ───────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(dem_synth, extent=extent_s, origin='lower', cmap='terrain',
               vmin=0, vmax=np.percentile(dem_synth, 99))
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Altitude (m)")
ax.set_title("(a) MNT conditionné", fontsize=11)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (b) Accumulation de flux (log) ────────────────────────────────────────────
ax = axes[1]
acc_plot = np.array(acc) if hasattr(acc, '__array__') else acc
acc_log  = np.log1p(acc_plot)
im2 = ax.imshow(acc_log, extent=extent_s, origin='lower', cmap='Blues',
                vmin=0, vmax=np.percentile(acc_log, 99))
cb2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
cb2.set_label("log(accumulation + 1)", fontsize=9)
ax.set_title("(b) Accumulation de flux (log)", fontsize=11)
ax.set_xlabel("Longitude")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (c) Réseau hydrographique + bassin versant ────────────────────────────────
ax = axes[2]
ax.imshow(dem_synth, extent=extent_s, origin='lower', cmap='terrain',
          vmin=0, vmax=np.percentile(dem_synth, 99), alpha=0.6)

stream_arr = np.array(streams) if hasattr(streams, '__array__') else streams
stream_rgba = np.zeros((*stream_arr.shape, 4))
stream_rgba[stream_arr, 0] = 0.0
stream_rgba[stream_arr, 2] = 0.8
stream_rgba[stream_arr, 3] = 0.9
ax.imshow(stream_rgba, extent=extent_s, origin='lower')

if catchment_arr is not None:
    catch_plot = np.array(catchment_arr) if hasattr(catchment_arr, '__array__') else catchment_arr
    catch_rgba = np.zeros((*catch_plot.shape, 4))
    catch_rgba[catch_plot.astype(bool), 0] = 1.0
    catch_rgba[catch_plot.astype(bool), 1] = 0.5
    catch_rgba[catch_plot.astype(bool), 3] = 0.25
    ax.imshow(catch_rgba, extent=extent_s, origin='lower')

ax.set_title(f"(c) Réseau hydrographique (seuil={STREAM_THRESHOLD})\n+ bassin versant", fontsize=10)
ax.set_xlabel("Longitude")
ax.grid(True, linestyle='--', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue',   alpha=0.8, label="Réseau hydrographique"),
    Patch(facecolor='orange', alpha=0.4, label="Bassin versant"),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8)

plt.tight_layout()
fig.savefig(DATA_DIR / "dem_pysheds_preprocessing.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée.")

---
## Section 4 — HAND (Height Above Nearest Drainage)

Le **HAND** (Hauteur au-dessus du drainage le plus proche) est un indice terrain dérivé du MNT qui quantifie la hauteur de chaque cellule au-dessus du réseau hydrographique le plus proche en termes de chemin d'écoulement.

**Propriétés clés :**
- HAND ≈ 0 m → zone inondable (fond de vallée, plaine alluviale)
- HAND élevé → zone hors d'inondation (versant, plateau)
- Proxy peu coûteux pour estimer l'extension des crues sans modélisation hydraulique complète

**Référence :** Nobre et al. (2011). *Height Above the Nearest Drainage – a hydrologically relevant new terrain model.* Journal of Hydrology.

In [ ]:
# ─── Calcul du HAND avec pysheds ──────────────────────────────────────────────
HAND_THRESHOLD = 500   # seuil d'accumulation définissant le drainage de référence

if PYSHEDS_AVAILABLE and fdir is not None:
    try:
        hand = grid.compute_hand(
            fdir=fdir,
            dem=dem_cdf,
            mask=acc > HAND_THRESHOLD,
            dirmap=dirmap
        )
        hand_arr = np.array(hand)
        hand_arr = np.where(hand_arr < 0, np.nan, hand_arr)
        print(f"HAND calculé — min={np.nanmin(hand_arr):.1f}m, max={np.nanmax(hand_arr):.1f}m")
    except Exception as e:
        print(f"[INFO] Calcul HAND pysheds : {e}")
        hand_arr = None
else:
    hand_arr = None

# ── HAND synthétique si pysheds non disponible ─────────────────────────────────
if hand_arr is None:
    print("[INFO] Calcul HAND synthétique.")
    # Approximation : HAND ≈ élévation relative au minimum local
    from scipy.ndimage import uniform_filter
    try:
        local_min = uniform_filter(dem_synth.astype(float), size=15)
        hand_arr  = np.clip(dem_synth - local_min, 0, None).astype(float)
        # Amplification dans les plaines (faible altitude)
        low_mask  = dem_synth < np.percentile(dem_synth, 30)
        hand_arr[low_mask] *= 0.4
        print(f"HAND synthétique — min={hand_arr.min():.1f}m, max={hand_arr.max():.1f}m")
    except ImportError:
        # Sans scipy
        hand_arr = np.clip(dem_synth - dem_synth.min(), 0, 50).astype(float)
        hand_arr = hand_arr / hand_arr.max() * 15
        print(f"HAND simplifié — min={hand_arr.min():.1f}m, max={hand_arr.max():.1f}m")

In [ ]:
# ─── Visualisation HAND avec classes d'inondabilité ───────────────────────────
# Classes de susceptibilité à l'inondation basées sur le HAND
hand_classes = [
    (0,  1,  '#d73027', 'HAND 0–1 m   (très élevée)'),
    (1,  3,  '#fc8d59', 'HAND 1–3 m   (élevée)'),
    (3,  5,  '#fee090', 'HAND 3–5 m   (modérée)'),
    (5,  10, '#e0f3f8', 'HAND 5–10 m  (faible)'),
    (10, np.nanmax(hand_arr) + 1 if hand_arr is not None else 200,
         '#4575b4', 'HAND > 10 m  (nulle)'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("HAND (Height Above Nearest Drainage) — Hérault", fontsize=14, fontweight='bold')

# ── Panneau gauche : HAND continu ─────────────────────────────────────────────
ax1 = axes[0]
hand_display = np.clip(hand_arr, 0, 15)
im_hand = ax1.imshow(
    hand_display, extent=extent_s, origin='lower',
    cmap='RdYlGn_r', vmin=0, vmax=15
)
cb = fig.colorbar(im_hand, ax=ax1, fraction=0.046, pad=0.04)
cb.set_label("HAND (m)", fontsize=10)
ax1.set_title("HAND continu (plafonné à 15 m)", fontsize=11)
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.grid(True, linestyle='--', alpha=0.3)

# ── Panneau droit : HAND discrétisé par classes ────────────────────────────────
ax2 = axes[1]
hand_class_img = np.zeros((*hand_arr.shape, 4))
for lo, hi, color, label in hand_classes:
    mask = (hand_arr >= lo) & (hand_arr < hi)
    rgba = mcolors.to_rgba(color)
    hand_class_img[mask] = rgba

ax2.imshow(hand_class_img, extent=extent_s, origin='lower')
ax2.set_title("Classes de susceptibilité à l'inondation\n(basé sur HAND)", fontsize=11)
ax2.set_xlabel("Longitude")
ax2.grid(True, linestyle='--', alpha=0.3)

legend_elements = [
    Patch(facecolor=color, label=label)
    for _, _, color, label in hand_classes
]
ax2.legend(handles=legend_elements, loc='upper left', fontsize=8,
           framealpha=0.85, title="Susceptibilité")

plt.tight_layout()
fig.savefig(DATA_DIR / "hand_inondabilite.png", dpi=150, bbox_inches='tight')
plt.show()

# Statistiques par classe
print("\nDistribution HAND par classe d'inondabilité :")
print(f"{'Classe':<28} {'Cellules':>10} {'% surface':>12}")
print("-" * 52)
total_valid = np.sum(~np.isnan(hand_arr))
for lo, hi, color, label in hand_classes:
    n = int(np.sum((hand_arr >= lo) & (hand_arr < hi)))
    pct = 100 * n / total_valid if total_valid > 0 else 0
    print(f"{label:<28} {n:>10,} {pct:>11.1f}%")

---
## Section 5 — Whitebox Tools

**WhiteboxTools** est une bibliothèque d'analyse géospatiale développée par l'Université de Guelph, spécialisée dans le traitement hydrologique avancé des MNTs.

**Avantages par rapport à pysheds :**
- **Breach depressions** (percée des dépressions) : préféré au remplissage pour les MNTs LiDAR haute résolution
- Algorithmes D-infinity (Tarboton 1997) pour des directions d'écoulement continues
- Large palette d'indices morphologiques (TWI, TRI, TPI, roughness…)
- Traitement des nuages de points LiDAR

> Installation : `pip install whitebox`

In [ ]:
# ─── Initialisation de WhiteboxTools ──────────────────────────────────────────
try:
    import whitebox
    wbt = whitebox.WhiteboxTools()
    wbt.set_working_dir(str(DATA_DIR.resolve()))
    wbt.verbose = False
    WBT_AVAILABLE = True
    print(f"WhiteboxTools version : {wbt.version()}")
    print(f"Répertoire de travail : {wbt.get_working_dir()}")
except ImportError:
    WBT_AVAILABLE = False
    print("[INFO] whitebox non installé. Exécutez : pip install whitebox")
    print("       Les exemples ci-dessous illustrent le code d'utilisation.")

In [ ]:
# ─── Breach Depressions (plus adapté aux MNTs LiDAR que le fill) ──────────────
#
# Le 'breach' (percée) crée un chemin minimal à travers les obstacles
# plutôt que de remplir entièrement la dépression.
# Avantage : préserve mieux la morphologie réelle du terrain.

if WBT_AVAILABLE and dem_temp_path.exists():
    breached_path = DATA_DIR / "dem_breached.tif"
    wbt.breach_depressions(
        dem=str(dem_temp_path),
        output=str(breached_path),
        max_depth=None,   # profondeur max de percée (None = illimitée)
        max_length=None,  # longueur max de percée
        flat_increment=None,
        fill_pits=True
    )
    print(f"Breach depressions terminé : {breached_path}")
else:
    print("[CODE] wbt.breach_depressions(")
    print("           dem='dem_herault.tif',")
    print("           output='dem_breached.tif',")
    print("           max_depth=None,")
    print("           fill_pits=True")
    print("       )")
    breached_path = None

In [ ]:
# ─── TWI (Topographic Wetness Index) ──────────────────────────────────────────
#
# TWI = ln(a / tan(β))
# où a = surface drainée spécifique (m²/m), β = pente locale (radians)
# Valeurs élevées → zones humides, susceptibles aux saturations et crues

if WBT_AVAILABLE and (breached_path or dem_temp_path.exists()):
    src_dem = breached_path if breached_path and breached_path.exists() else dem_temp_path

    # Étape 1 : Accumulation de flux D8
    flow_accum_path = DATA_DIR / "flow_accum_d8.tif"
    wbt.d8_flow_accumulation(
        i=str(src_dem),
        output=str(flow_accum_path),
        out_type='specific contributing area'
    )

    # Étape 2 : Pente
    slope_path = DATA_DIR / "slope.tif"
    wbt.slope(dem=str(src_dem), output=str(slope_path), units='radians')

    # Étape 3 : TWI
    twi_path = DATA_DIR / "twi.tif"
    wbt.wetness_index(
        sca=str(flow_accum_path),
        slope=str(slope_path),
        output=str(twi_path)
    )
    print(f"TWI calculé : {twi_path}")
else:
    print("[CODE] Calcul TWI avec WhiteboxTools :")
    print("       # 1. Accumulation de flux")
    print("       wbt.d8_flow_accumulation(i='dem_breached.tif', output='flow_accum.tif',")
    print("                                out_type='specific contributing area')")
    print("       # 2. Pente")
    print("       wbt.slope(dem='dem_breached.tif', output='slope.tif', units='radians')")
    print("       # 3. TWI")
    print("       wbt.wetness_index(sca='flow_accum.tif', slope='slope.tif', output='twi.tif')")
    twi_path = None

In [ ]:
# ─── D-infinity Flow Accumulation (Tarboton 1997) ─────────────────────────────
#
# Contrairement au D8 (8 directions discrètes), D-inf distribue le flux
# de façon continue entre deux directions voisines (angle libre).
# Résultat : accumulation plus réaliste sur les versants convexes.

if WBT_AVAILABLE and (breached_path or dem_temp_path.exists()):
    src_dem = breached_path if breached_path and breached_path.exists() else dem_temp_path
    dinf_path = DATA_DIR / "flow_accum_dinf.tif"
    wbt.d_inf_flow_accumulation(
        i=str(src_dem),
        output=str(dinf_path),
        out_type='specific contributing area',
        convergence=5.0
    )
    print(f"D-inf accumulation calculée : {dinf_path}")
else:
    print("[CODE] wbt.d_inf_flow_accumulation(")
    print("           i='dem_breached.tif',")
    print("           output='flow_accum_dinf.tif',")
    print("           out_type='specific contributing area'")
    print("       )")
    dinf_path = None

In [ ]:
# ─── Visualisation TWI (réel ou synthétique) ──────────────────────────────────
# Chargement du TWI si disponible, sinon calcul analytique
if twi_path and twi_path.exists():
    try:
        import rasterio
        with rasterio.open(twi_path) as src:
            twi_arr = src.read(1).astype(float)
            twi_arr[twi_arr < -1e10] = np.nan
    except ImportError:
        twi_arr = None
else:
    twi_arr = None

if twi_arr is None:
    # TWI synthétique : TWI ≈ ln((acc+1) / (tan(slope)+0.01))
    # Calcul de la pente par gradient numérique
    dy, dx = np.gradient(dem_synth, 30, 30)   # résolution 30m
    slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
    twi_arr = np.log((acc + 1) / (np.tan(slope_rad) + 0.01))
    twi_arr = np.clip(twi_arr, 0, 20)
    print(f"TWI synthétique — min={twi_arr.min():.2f}, max={twi_arr.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Topographic Wetness Index (TWI) — WhiteboxTools", fontsize=14, fontweight='bold')

# ── TWI carte continue ────────────────────────────────────────────────────────
ax1 = axes[0]
im_twi = ax1.imshow(
    twi_arr, extent=extent_s, origin='lower',
    cmap='Blues', vmin=np.nanpercentile(twi_arr, 2), vmax=np.nanpercentile(twi_arr, 98)
)
fig.colorbar(im_twi, ax=ax1, fraction=0.046, pad=0.04, label="TWI")
ax1.set_title("TWI = ln(a / tan β)\nValeurs élevées = zones humides", fontsize=11)
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.grid(True, linestyle='--', alpha=0.3)

# ── Histogramme TWI ───────────────────────────────────────────────────────────
ax2 = axes[1]
twi_flat = twi_arr[~np.isnan(twi_arr)].ravel()
n, bins, patches = ax2.hist(twi_flat, bins=60, color='steelblue', edgecolor='none', alpha=0.8)

# Colorisation par valeur TWI
norm = plt.Normalize(bins.min(), bins.max())
cm   = plt.cm.Blues
for patch, left_edge in zip(patches, bins[:-1]):
    patch.set_facecolor(cm(norm(left_edge)))

# Seuils indicatifs
for thresh, label, col in [(6, 'TWI=6 (humide)', 'orange'), (10, 'TWI=10 (saturé)', 'red')]:
    ax2.axvline(thresh, color=col, linestyle='--', linewidth=1.5, label=label)

ax2.set_xlabel("TWI", fontsize=11)
ax2.set_ylabel("Nombre de cellules", fontsize=11)
ax2.set_title("Distribution du TWI", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.4)

p_wet = 100 * np.sum(twi_flat > 6) / len(twi_flat)
ax2.text(0.97, 0.95, f"{p_wet:.1f}% cellules TWI > 6",
         transform=ax2.transAxes, ha='right', va='top',
         fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
fig.savefig(DATA_DIR / "twi_whitebox.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure TWI sauvegardée.")

---
## Section 6 — Synthèse : comparaison des DEMs et méthodes

### 6.1 — Tableau de comparaison des sources DEM

In [ ]:
# ─── Tableau comparatif final des sources DEM ─────────────────────────────────
print("=" * 90)
print(" COMPARAISON DES SOURCES DEM POUR L'ANALYSE DES CRUES — HÉRAULT (34)")
print("=" * 90)

table_dem = [
    {
        "Source"         : "Copernicus DEM GLO-30",
        "Résolution"     : "~30 m",
        "Précision (Z)"  : "±4 m",
        "Couverture"     : "Mondiale",
        "Accès"          : "Gratuit (API)",
        "Usage crues"    : "Régional, bassin versant",
        "Limites"        : "Résolution insuffisante intraurbain",
    },
    {
        "Source"         : "IGN RGE ALTI 1m",
        "Résolution"     : "1 m",
        "Précision (Z)"  : "±0.2 m",
        "Couverture"     : "France entière",
        "Accès"          : "Gratuit (WCS)",
        "Usage crues"    : "Local, hydraulique fine",
        "Limites"        : "Volume de données élevé",
    },
    {
        "Source"         : "IGN LiDAR HD 50cm",
        "Résolution"     : "0.5 m",
        "Précision (Z)"  : "±0.1 m",
        "Couverture"     : "France (en cours)",
        "Accès"          : "Gratuit (LAZ/COPC)",
        "Usage crues"    : "Très fin, MNS, levées",
        "Limites"        : "Couverture incomplète 2024",
    },
]

fields = list(table_dem[0].keys())
widths = {f: max(len(f), max(len(row[f]) for row in table_dem)) + 2 for f in fields}

header = " | ".join(f"{f:<{widths[f]}}" for f in fields)
sep    = "-+-".join("-" * widths[f] for f in fields)
print(header)
print(sep)
for row in table_dem:
    print(" | ".join(f"{row[f]:<{widths[f]}}" for f in fields))
print()

### 6.2 — Tableau de comparaison des outils de traitement

In [ ]:
# ─── Tableau comparatif pysheds vs WhiteboxTools ──────────────────────────────
print("=" * 78)
print(" COMPARAISON DES OUTILS DE PRÉTRAITEMENT DEM")
print("=" * 78)

tool_comparison = [
    {
        "Fonctionnalité"       : "Remplissage dépressions",
        "pysheds"              : "fill_depressions() ✓",
        "WhiteboxTools"        : "fill_depressions() ✓",
    },
    {
        "Fonctionnalité"       : "Breach depressions",
        "pysheds"              : "— Non disponible",
        "WhiteboxTools"        : "breach_depressions() ✓",
    },
    {
        "Fonctionnalité"       : "Directions D8",
        "pysheds"              : "flowdir() ✓",
        "WhiteboxTools"        : "d8_pointer() ✓",
    },
    {
        "Fonctionnalité"       : "Directions D-infinity",
        "pysheds"              : "— Non disponible",
        "WhiteboxTools"        : "d_inf_flow_accum() ✓",
    },
    {
        "Fonctionnalité"       : "Accumulation de flux",
        "pysheds"              : "accumulation() ✓",
        "WhiteboxTools"        : "d8_flow_accumulation() ✓",
    },
    {
        "Fonctionnalité"       : "HAND",
        "pysheds"              : "compute_hand() ✓",
        "WhiteboxTools"        : "elevation_above_stream() ✓",
    },
    {
        "Fonctionnalité"       : "TWI",
        "pysheds"              : "— Manuel (post-traitement)",
        "WhiteboxTools"        : "wetness_index() ✓",
    },
    {
        "Fonctionnalité"       : "Extraction bassin versant",
        "pysheds"              : "catchment() ✓",
        "WhiteboxTools"        : "watershed() ✓",
    },
    {
        "Fonctionnalité"       : "Traitement LiDAR",
        "pysheds"              : "— Non",
        "WhiteboxTools"        : "Oui (lidar_ground_filter, etc.)",
    },
    {
        "Fonctionnalité"       : "Performance",
        "pysheds"              : "Moyenne (Python pur)",
        "WhiteboxTools"        : "Élevée (Rust backend)",
    },
    {
        "Fonctionnalité"       : "Intégration Python",
        "pysheds"              : "Natif (xarray/numpy)",
        "WhiteboxTools"        : "Via wrapper Python + I/O fichiers",
    },
]

fields_t = ["Fonctionnalité", "pysheds", "WhiteboxTools"]
widths_t = {f: max(len(f), max(len(row[f]) for row in tool_comparison)) + 2 for f in fields_t}

header_t = " | ".join(f"{f:<{widths_t[f]}}" for f in fields_t)
sep_t    = "-+-".join("-" * widths_t[f] for f in fields_t)
print(header_t)
print(sep_t)
for row in tool_comparison:
    print(" | ".join(f"{row[f]:<{widths_t[f]}}" for f in fields_t))

### 6.3 — Recommandations pour la modélisation hydraulique

In [ ]:
# ─── Recommandations et conclusions ───────────────────────────────────────────
recommendations = """
╔══════════════════════════════════════════════════════════════════════════════╗
║         RECOMMANDATIONS POUR L'ANALYSE DES CRUES — HÉRAULT (34)            ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. CHOIX DU DEM SELON L'ÉCHELLE                                             ║
║     • Bassin versant régional (> 100 km²) → Copernicus GLO-30               ║
║     • Analyse locale / plaine inondable    → IGN RGE ALTI 1m                ║
║     • Modélisation hydraulique urbaine     → IGN LiDAR HD 0.5m              ║
║                                                                              ║
║  2. CONDITIONNEMENT HYDROLOGIQUE                                             ║
║     • Pour MNTs à ≥ 5m : fill_pits + fill_depressions (pysheds)            ║
║     • Pour LiDAR (1m ou 50cm) : PRÉFÉRER breach_depressions (whitebox)     ║
║       → Le remplissage sur LiDAR génère des plaines artificielles           ║
║     • Toujours utiliser resolve_flats avant le calcul des flux              ║
║                                                                              ║
║  3. INDICES DÉRIVÉS POUR LES CRUES                                          ║
║     • HAND    : délimitation rapide des zones inondables sans simulation     ║
║     • TWI     : saturation des sols, probabilité de ruissellement           ║
║     • D-inf   : accumulation de flux plus réaliste (versus D8)              ║
║     • TRI/TPI : identification des corridors torrentiels                    ║
║                                                                              ║
║  4. CHAÎNE RECOMMANDÉE POUR L'HÉRAULT                                       ║
║     LiDAR HD (50cm) → breach (WhiteboxTools) → D-inf accumulation          ║
║     → HAND + TWI → délimitation zones inondables → modèle hydraulique 2D   ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(recommendations)

In [ ]:
# ─── Figure récapitulative : pipeline MNT → analyse crues ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Pipeline MNT → Analyse des crues — {DEPT_NAME} (dép. {DEPT_CODE})",
    fontsize=15, fontweight='bold', y=1.01
)

extent_s = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]

# ── (1) DEM brut ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
im = ax.imshow(dem_synth, extent=extent_s, origin='lower',
               cmap='terrain', vmin=0, vmax=900)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="m")
ax.set_title("1. DEM brut (Copernicus 30m)", fontsize=10)
ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (2) Hillshade ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
ls2 = LightSource(azdeg=315, altdeg=45)
rgb2 = ls2.shade(dem_synth, cmap=plt.cm.terrain, blend_mode='overlay',
                  vmin=0, vmax=900, vert_exag=3, dx=30, dy=30)
ax.imshow(rgb2, extent=extent_s, origin='lower')
ax.set_title("2. Ombrage (hillshade)", fontsize=10)
ax.set_xlabel("Lon")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (3) Accumulation de flux ───────────────────────────────────────────────────
ax = axes[0, 2]
acc_plot = np.array(acc) if hasattr(acc, '__array__') else acc
im3 = ax.imshow(np.log1p(acc_plot), extent=extent_s, origin='lower',
                cmap='Blues', vmin=0, vmax=np.percentile(np.log1p(acc_plot), 99))
fig.colorbar(im3, ax=ax, fraction=0.046, pad=0.04, label="log(acc+1)")
ax.set_title("3. Accumulation de flux (D8)", fontsize=10)
ax.set_xlabel("Lon")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (4) Réseau hydrographique ─────────────────────────────────────────────────
ax = axes[1, 0]
ax.imshow(dem_synth, extent=extent_s, origin='lower',
          cmap='Greens', alpha=0.5, vmin=0, vmax=900)
stream_mask = acc_plot > STREAM_THRESHOLD
st_rgba = np.zeros((*stream_mask.shape, 4))
st_rgba[stream_mask] = [0, 0.3, 0.9, 0.9]
ax.imshow(st_rgba, extent=extent_s, origin='lower')
ax.set_title(f"4. Réseau hydrographique (seuil={STREAM_THRESHOLD})", fontsize=10)
ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (5) HAND ─────────────────────────────────────────────────────────────────
ax = axes[1, 1]
im5 = ax.imshow(np.clip(hand_arr, 0, 15), extent=extent_s, origin='lower',
                cmap='RdYlGn_r', vmin=0, vmax=15)
fig.colorbar(im5, ax=ax, fraction=0.046, pad=0.04, label="HAND (m)")
ax.set_title("5. HAND (inondabilité)", fontsize=10)
ax.set_xlabel("Lon")
ax.grid(True, linestyle='--', alpha=0.3)

# ── (6) TWI ──────────────────────────────────────────────────────────────────
ax = axes[1, 2]
im6 = ax.imshow(
    twi_arr, extent=extent_s, origin='lower', cmap='Blues',
    vmin=np.nanpercentile(twi_arr, 2), vmax=np.nanpercentile(twi_arr, 98)
)
fig.colorbar(im6, ax=ax, fraction=0.046, pad=0.04, label="TWI")
ax.set_title("6. TWI (zones humides)", fontsize=10)
ax.set_xlabel("Lon")
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
fig.savefig(DATA_DIR / "dem_pipeline_synthese.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure de synthèse sauvegardée.")

In [ ]:
# ─── Résumé des fichiers produits ─────────────────────────────────────────────
print("\nFichiers produits dans", DATA_DIR.resolve())
print("-" * 60)
output_files = [
    ("dem_cop30_visualisation.png" , "Carte DEM COP30 + hillshade"),
    ("dem_pysheds_preprocessing.png", "Prétraitement pysheds"),
    ("hand_inondabilite.png"        , "HAND et classes d'inondabilité"),
    ("twi_whitebox.png"             , "TWI WhiteboxTools"),
    ("dem_pipeline_synthese.png"    , "Pipeline complet MNT → crues"),
    ("dem_synth_herault.tif"        , "DEM synthétique de démonstration"),
]
for fname, desc in output_files:
    fpath = DATA_DIR / fname
    status = "OK" if fpath.exists() else "absent"
    print(f"  [{status:>6}] {fname:<40} {desc}")

print()
print("Notebook terminé. Prochaine étape : 02b_slope_morphology.ipynb")